# 🏆 OlympiQ — Sistem Seleksi Olimpiade Matematika
### Data Mining · Random Forest Classifier

**Kelompok:** [Isi Nama Kelompok]

**Prodi:** Pendidikan Matematika

---

## 📋 Deskripsi Proyek

Notebook ini membangun sistem klasifikasi berbasis machine learning untuk **seleksi tahap awal calon peserta olimpiade matematika**. Sistem menggunakan 6 fitur skor kompetensi siswa untuk memprediksi kesiapan olimpiade ke dalam 3 kelas:

- 🏆 **Siap Olimpiade** — Siap mengikuti seleksi
- ⚡ **Potensial** — Berpotensi namun perlu pembinaan
- ❌ **Tidak Siap** — Perlu penguatan kompetensi dasar

**Algoritma:** Random Forest Classifier

**Dataset:** 30.000 data siswa dengan 6 fitur skor


## 📦 1. Instalasi Library

In [ ]:
# Install library yang diperlukan
!pip install -q scikit-learn pandas numpy openpyxl joblib plotly xlsxwriter
print("✅ Semua library berhasil diinstall!")

## 📥 2. Import Library & Konfigurasi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, ConfusionMatrixDisplay)
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

# Konfigurasi matplotlib untuk tampilan lebih baik
plt.rcParams["figure.facecolor"] = "#0d1226"
plt.rcParams["axes.facecolor"]   = "#12172b"
plt.rcParams["text.color"]        = "white"
plt.rcParams["axes.labelcolor"]   = "white"
plt.rcParams["xtick.color"]       = "white"
plt.rcParams["ytick.color"]       = "white"
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("✅ Import selesai!")

## 📂 3. Upload & Load Dataset

In [ ]:
# Upload dataset dari komputer lokal
from google.colab import files

print("📂 Silakan upload file dataset.xlsx")
uploaded = files.upload()

# Load dataset
DATA_FILE = list(uploaded.keys())[0]
df = pd.read_excel(DATA_FILE)

# Rename kolom sesuai standar sistem
FEATURE_COLS = ["NUM_ALJ", "NUM_GEO", "NUM_BIL", "NUM_DAT", "NUM_L3", "LIT"]
df.columns = FEATURE_COLS

print(f"✅ Dataset berhasil dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head(10)

## 🔍 4. Eksplorasi Data (EDA)

In [ ]:
# Statistik deskriptif
print("=" * 55)
print("  STATISTIK DESKRIPTIF DATASET")
print("=" * 55)
df.describe().round(2)

In [ ]:
# Cek missing values
print("Missing Values per Kolom:")
missing = df.isnull().sum()
for col, mv in missing.items():
    pct = mv/len(df)*100
    print(f"  {col}: {mv} ({pct:.2f}%)")

# Drop missing values
df_clean = df.dropna().reset_index(drop=True)
print(f"
Data valid setelah drop NA: {len(df_clean):,} baris")

In [ ]:
# Visualisasi distribusi skor tiap fitur
FEATURE_NAMES = {
    "NUM_ALJ": "Numerasi Aljabar",
    "NUM_GEO": "Numerasi Geometri",
    "NUM_BIL": "Numerasi Bilangan",
    "NUM_DAT": "Data & Ketidakpastian",
    "NUM_L3":  "Skor Menalar",
    "LIT":     "Literasi"
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
colors = ["#63b3ed","#10b981","#f59e0b","#ef4444","#a78bfa","#fb7185"]
axes = axes.flatten()

for i, (col, color) in enumerate(zip(FEATURE_COLS, colors)):
    axes[i].hist(df_clean[col], bins=40, color=color, alpha=0.8, edgecolor="none")
    axes[i].axvline(df_clean[col].mean(), color="white", linestyle="--", lw=1.5,
                    label=f"Mean: {df_clean[col].mean():.1f}")
    axes[i].set_title(FEATURE_NAMES[col], fontsize=12, fontweight="bold", pad=10)
    axes[i].set_xlabel("Skor")
    axes[i].set_ylabel("Frekuensi")
    axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.15)

plt.suptitle("Distribusi Skor Tiap Komponen", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("distribusi_skor.png", dpi=150, bbox_inches="tight", facecolor="#0d1226")
plt.show()
print("✅ Distribusi skor divisualisasikan!")

In [ ]:
# Heatmap korelasi antar fitur
fig, ax = plt.subplots(figsize=(9, 7))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    square=True, linewidths=0.5, ax=ax, annot_kws={"size":10},
    xticklabels=[FEATURE_NAMES[c] for c in FEATURE_COLS],
    yticklabels=[FEATURE_NAMES[c] for c in FEATURE_COLS]
)
ax.set_title("Korelasi Antar Fitur Kompetensi Siswa", fontsize=13, fontweight="bold", pad=12)
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("heatmap_korelasi.png", dpi=150, bbox_inches="tight", facecolor="#0d1226")
plt.show()

## 🏷️ 5. Pembuatan Label (Klasifikasi)

In [ ]:
# Konfigurasi bobot tiap fitur (skenario berdasarkan kepentingan kompetensi olimpiade)
WEIGHTS = {
    "NUM_ALJ": 0.20,  # Aljabar: bobot tinggi karena fundamental olimpiade
    "NUM_GEO": 0.15,  # Geometri
    "NUM_BIL": 0.20,  # Bilangan: bobot tinggi karena dominan di soal olimpiade
    "NUM_DAT": 0.15,  # Data & Probabilitas
    "NUM_L3":  0.15,  # Penalaran
    "LIT":     0.15   # Literasi
}

# Hitung skor total berbobot
df_clean["TOTAL_SCORE"] = sum(df_clean[col] * w for col, w in WEIGHTS.items())

# Tentukan threshold berdasarkan percentile distribusi
p33 = df_clean["TOTAL_SCORE"].quantile(0.33)
p66 = df_clean["TOTAL_SCORE"].quantile(0.66)

print(f"Threshold Tidak Siap    : < {p33:.2f}")
print(f"Threshold Potensial     : {p33:.2f} - {p66:.2f}")
print(f"Threshold Siap Olimpiade: >= {p66:.2f}")

# Buat label
CLASS_MAP = {0: "Tidak Siap", 1: "Potensial", 2: "Siap Olimpiade"}

df_clean["LABEL"] = df_clean["TOTAL_SCORE"].apply(
    lambda s: 2 if s >= p66 else (1 if s >= p33 else 0)
)

print("
Distribusi Label:")
for label, count in df_clean["LABEL"].value_counts().sort_index().items():
    print(f"  {CLASS_MAP[label]}: {count:,} ({count/len(df_clean)*100:.1f}%)")

In [ ]:
# Visualisasi distribusi label
label_counts = df_clean["LABEL"].value_counts().sort_index()
label_names  = [CLASS_MAP[i] for i in label_counts.index]
colors_pie   = ["#ef4444", "#f59e0b", "#10b981"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
wedges, texts, autotexts = ax1.pie(
    label_counts.values,
    labels=label_names,
    colors=colors_pie,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.7,
    wedgeprops={"edgecolor": "#0d1226", "linewidth": 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_color("white")
    at.set_fontweight("bold")
ax1.set_title("Distribusi Status Kesiapan Olimpiade", fontsize=12, fontweight="bold")

# Bar chart skor total
ax2.hist([df_clean.loc[df_clean.LABEL==0, "TOTAL_SCORE"],
          df_clean.loc[df_clean.LABEL==1, "TOTAL_SCORE"],
          df_clean.loc[df_clean.LABEL==2, "TOTAL_SCORE"]],
         bins=50, stacked=True, color=colors_pie, alpha=0.85,
         label=["Tidak Siap", "Potensial", "Siap Olimpiade"])
ax2.axvline(p33, color="white", linestyle="--", lw=1.5, label=f"P33 = {p33:.1f}")
ax2.axvline(p66, color="#63b3ed", linestyle="--", lw=1.5, label=f"P66 = {p66:.1f}")
ax2.set_xlabel("Skor Total")
ax2.set_ylabel("Jumlah Siswa")
ax2.set_title("Distribusi Skor Total per Kelas", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.1)

plt.tight_layout()
plt.savefig("distribusi_label.png", dpi=150, bbox_inches="tight", facecolor="#0d1226")
plt.show()

## 🤖 6. Training Model Random Forest

In [ ]:
# Split Data
X = df_clean[FEATURE_COLS]
y = df_clean["LABEL"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisasi fitur
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Data latih : {len(X_train):,}")
print(f"Data uji   : {len(X_test):,}")

# Training Random Forest
print("
⏳ Melatih model Random Forest...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)
print("✅ Model berhasil dilatih!")

## 📊 7. Evaluasi Model

In [ ]:
# Prediksi & Akurasi
y_pred = rf.predict(X_test_sc)
acc    = accuracy_score(y_test, y_pred)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train_sc, y_train, cv=cv, scoring="accuracy")

print("=" * 55)
print("  HASIL EVALUASI MODEL")
print("=" * 55)
print(f"  Akurasi Test Set   : {acc:.4f} ({acc*100:.2f}%)")
print(f"  CV Mean (5-fold)   : {cv_scores.mean():.4f}")
print(f"  CV Std             : {cv_scores.std():.4f}")
print()
print(classification_report(
    y_test, y_pred,
    target_names=["Tidak Siap", "Potensial", "Siap Olimpiade"]
))

In [ ]:
# Visualisasi Confusion Matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Tidak Siap", "Potensial", "Siap Olimpiade"]
)
disp.plot(ax=ax1, cmap="Blues", colorbar=False)
ax1.set_title("Confusion Matrix", fontsize=12, fontweight="bold")
ax1.tick_params(axis="x", rotation=20)

# Feature Importance
importances = rf.feature_importances_
feat_names  = [FEATURE_NAMES[c] for c in FEATURE_COLS]
sorted_idx  = np.argsort(importances)
colors_fi   = ["#63b3ed"] * len(importances)

ax2.barh([feat_names[i] for i in sorted_idx],
         [importances[i]*100 for i in sorted_idx],
         color=colors_fi, edgecolor="none", alpha=0.9)
ax2.set_xlabel("Importance (%)")
ax2.set_title("Feature Importance — Random Forest", fontsize=12, fontweight="bold")
for i, (idx, val) in enumerate(zip(sorted_idx, [importances[i]*100 for i in sorted_idx])):
    ax2.text(val + 0.2, i, f"{val:.1f}%", va="center", fontsize=9, color="white")
ax2.grid(axis="x", alpha=0.1)

plt.tight_layout()
plt.savefig("evaluasi_model.png", dpi=150, bbox_inches="tight", facecolor="#0d1226")
plt.show()

## 💾 8. Simpan Model & Deploy ke GitHub + Streamlit

In [ ]:
# Buat folder model
os.makedirs("model", exist_ok=True)

# Simpan model & artefak
joblib.dump(rf,                           "model/rf_model.pkl")
joblib.dump(scaler,                       "model/scaler.pkl")
joblib.dump({"p33": p33, "p66": p66},   "model/thresholds.pkl")

meta = {
    "accuracy":           float(acc),
    "cv_mean":            float(cv_scores.mean()),
    "cv_std":             float(cv_scores.std()),
    "n_train":            int(len(X_train)),
    "n_test":             int(len(X_test)),
    "feature_cols":       FEATURE_COLS,
    "feature_names":      FEATURE_NAMES,
    "weights":            WEIGHTS,
    "thresholds":         {"p33": float(p33), "p66": float(p66)},
    "feature_importance": dict(zip(FEATURE_COLS, rf.feature_importances_.tolist()))
}
joblib.dump(meta, "model/model_meta.pkl")

print("✅ Model berhasil disimpan!")
print("File yang disimpan:")
for f in os.listdir("model"):
    size = os.path.getsize(f"model/{f}") / 1024
    print(f"  model/{f}  ({size:.1f} KB)")

In [ ]:
# Download semua file model dari Colab
from google.colab import files
import zipfile

# Zip folder model
with zipfile.ZipFile("model_olimpiade.zip", "w") as zf:
    for f in os.listdir("model"):
        zf.write(f"model/{f}")
    for img in ["distribusi_skor.png", "heatmap_korelasi.png",
                "distribusi_label.png", "evaluasi_model.png"]:
        if os.path.exists(img):
            zf.write(img)

files.download("model_olimpiade.zip")
print("✅ File berhasil didownload! Extract dan letakkan di folder model/ proyek kamu.")

## 🧪 9. Demo Prediksi Manual

Coba prediksi untuk beberapa siswa secara manual:

In [ ]:
# Prediksi manual untuk beberapa siswa contoh
def predict_student(nama, num_alj, num_geo, num_bil, num_dat, num_l3, lit):
    scores = {
        "NUM_ALJ": num_alj, "NUM_GEO": num_geo, "NUM_BIL": num_bil,
        "NUM_DAT": num_dat, "NUM_L3": num_l3,  "LIT": lit
    }
    X_input = pd.DataFrame([scores])[FEATURE_COLS]
    X_scaled = scaler.transform(X_input)
    label  = rf.predict(X_scaled)[0]
    proba  = rf.predict_proba(X_scaled)[0]
    total  = sum(scores[c] * WEIGHTS[c] for c in FEATURE_COLS)

    emoji = {2: "🏆", 1: "⚡", 0: "❌"}[label]
    print(f"  Nama   : {nama}")
    print(f"  Status : {emoji} {CLASS_MAP[label]}  (Skor Total: {total:.2f})")
    print(f"  Prob   : Tidak Siap={proba[0]*100:.1f}% | Potensial={proba[1]*100:.1f}% | Siap Olimpiade={proba[2]*100:.1f}%")
    print()

print("=" * 55)
print("  DEMO PREDIKSI SISWA")
print("=" * 55)

predict_student("Andi Budi Santoso",    95.0, 91.0, 88.5, 92.0, 89.0, 94.0)
predict_student("Rina Cahyani",         67.0, 70.5, 62.0, 68.0, 65.5, 72.0)
predict_student("Doni Prasetyo",        45.0, 38.0, 41.5, 50.0, 42.0, 55.0)